# Wanderbricks Users

**Dataset:** `samples.wanderbricks.users`

**Difficulty:** Easy

**Topics:** filter, groupBy, boolean columns, date

In [0]:
from pyspark.sql import functions as F, types as T

## Learn — Boolean Columns and Grouping

| Function | What it does |
|----------|-------------|
| `df.filter(F.col("bool_col") == True)` | Filter on a boolean column |
| `df.filter(F.col("bool_col"))` | Shorthand for `== True` |
| `F.when(condition, value).otherwise(other)` | Conditional column expression |
| `F.year("col")`, `F.datediff(end, start)` | Date extraction/comparison |
| `df.groupBy(col).agg(F.sum(F.col("bool").cast("int")))` | Count True values in a boolean column |

**Docs:** [PySpark Functions](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/functions.html) · [PySpark Functions](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/functions.html)

In [0]:
# Run this example first -- then solve the problems below.
# NOTE: this example is not a solution to any problem

df = spark.table("samples.wanderbricks.users")

# Explore the schema -- note the boolean columns
df.printSchema()

# Count users by user type (not is_business -- that's for the problems)
df.groupBy("user_type").count().show()

# How many users registered in each year?
df.groupBy(F.year("created_at").alias("reg_year")).count().orderBy("reg_year").show()

## Problem 1

Count the number of users per **country** and return the top 10 countries
by user count. Load `samples.wanderbricks.users`.

**Expected output columns:**
- `country` - country name
- `user_count` - number of users in that country (top 10)

In [0]:
# Problem 1 - write your solution here
# Assign your result to: result_1

result_1 = spark.read.table("samples.wanderbricks.users").groupBy("country").agg(
    F.count("*").alias("user_count")
).orderBy(F.col("user_count").desc()).limit(10)

In [0]:
# ── Tests for Problem 1 ──────────────────────────────────────────
assert result_1 is not None, "result_1 is None - did you forget to assign your DataFrame?"
assert hasattr(result_1, 'columns'), "result_1 must be a Spark DataFrame"
cols = [c.lower() for c in result_1.columns]
assert 'country' in cols, "Missing column: country"
assert 'user_count' in cols, "Missing column: user_count"
assert len(cols) == 2, f"Expected exactly 2 columns, got {len(cols)}: {cols}"
cnt = result_1.count()
assert cnt == 10, f"Expected exactly 10 rows (top 10), got {cnt}"
counts = [r['user_count'] for r in result_1.collect()]
assert all(c > 0 for c in counts), "All user_count values must be positive"
print(f"Problem 1 passed ✓  ({cnt} rows)")

## Problem 2

Count **business users vs individual users** using the boolean `is_business`
column. Group by `is_business` to see how many users fall into each category.

**Expected output columns:**
- `is_business` - `true` for business accounts, `false` for individual
- `user_count` - number of users in that category

In [0]:
# Problem 2 - write your solution here
# Assign your result to: result_2

result_2 = df.groupBy("is_business").agg(
    F.count("*").alias("user_count")
)

In [0]:
# ── Tests for Problem 2 ──────────────────────────────────────────
assert result_2 is not None, "result_2 is None - did you forget to assign your DataFrame?"
assert hasattr(result_2, 'columns'), "result_2 must be a Spark DataFrame"
cols = [c.lower() for c in result_2.columns]
assert 'is_business' in cols, "Missing column: is_business"
assert 'user_count' in cols, "Missing column: user_count"
assert len(cols) == 2, f"Expected exactly 2 columns, got {len(cols)}: {cols}"
cnt = result_2.count()
assert cnt in (1, 2), f"Expected 1 or 2 rows (true/false), got {cnt}"
counts = [r['user_count'] for r in result_2.collect()]
assert all(c > 0 for c in counts), "All user_count values must be positive"
print(f"Problem 2 passed ✓  ({cnt} rows)")

## Problem 3

Count users who **joined each year** by extracting the year from `created_at`.
Use `F.year()` on the timestamp column. Sort by year ascending.

**Expected output columns:**
- `join_year` - the calendar year the user account was created
- `user_count` - number of users who joined in that year (sorted ascending)

In [0]:
# Problem 3 - write your solution here
# Assign your result to: result_3

result_3 = df.groupBy(
    F.year("created_at").alias("join_year")
).agg(
    F.count("*").alias("user_count")
).orderBy(F.col("join_year"))

In [0]:
# ── Tests for Problem 3 ──────────────────────────────────────────
assert result_3 is not None, "result_3 is None - did you forget to assign your DataFrame?"
assert hasattr(result_3, 'columns'), "result_3 must be a Spark DataFrame"
cols = [c.lower() for c in result_3.columns]
assert 'join_year' in cols, "Missing column: join_year"
assert 'user_count' in cols, "Missing column: user_count"
assert len(cols) == 2, f"Expected exactly 2 columns, got {len(cols)}: {cols}"
cnt = result_3.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
years = [r['join_year'] for r in result_3.collect()]
assert years == sorted(years), "Results must be sorted by join_year ascending"
assert all(y > 2000 for y in years), "All join_year values should be > 2000"
assert all(r['user_count'] > 0 for r in result_3.collect()), "All user counts must be positive"
print(f"Problem 3 passed ✓  ({cnt} rows)")

## Problem 4

Count users per **user type** (`user_type` column).
This reveals the distribution of user roles on the platform.

**Expected output columns:**
- `user_type` - the type of user account
- `count` - number of users with that type

In [0]:
# Problem 4 - write your solution here
# Assign your result to: result_4

result_4 = df.groupBy("user_type").count()

In [0]:
# ── Tests for Problem 4 ──────────────────────────────────────────
assert result_4 is not None, "result_4 is None - did you forget to assign your DataFrame?"
assert hasattr(result_4, 'columns'), "result_4 must be a Spark DataFrame"
cols = [c.lower() for c in result_4.columns]
assert 'user_type' in cols, "Missing column: user_type"
assert 'count' in cols, "Missing column: count"
assert len(cols) == 2, f"Expected exactly 2 columns, got {len(cols)}: {cols}"
cnt = result_4.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
counts = [r['count'] for r in result_4.collect()]
assert all(c > 0 for c in counts), "All count values must be positive"
print(f"Problem 4 passed ✓  ({cnt} rows)")

## Problem 5

Find users where **`company_name` is not null** - these are verified business
users who have provided their company information.

**Expected output columns:**
- `user_id` - user identifier
- `name` - user's display name
- `country` - user's country
- `company_name` - the company name (must not be null)

In [0]:
# Problem 5 - write your solution here
# Assign your result to: result_5

result_5 = df.filter(~F.col("company_name").isNull()).select(
    "user_id",
    "name",
    "country",
    "company_name"
)

In [0]:
# ── Tests for Problem 5 ──────────────────────────────────────────
assert result_5 is not None, "result_5 is None - did you forget to assign your DataFrame?"
assert hasattr(result_5, 'columns'), "result_5 must be a Spark DataFrame"
cols = [c.lower() for c in result_5.columns]
assert 'user_id' in cols, "Missing column: user_id"
assert 'name' in cols, "Missing column: name"
assert 'country' in cols, "Missing column: country"
assert 'company_name' in cols, "Missing column: company_name"
assert len(cols) == 4, f"Expected exactly 4 columns, got {len(cols)}: {cols}"
cnt = result_5.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
null_count = result_5.filter(F.col('company_name').isNull()).count()
assert null_count == 0, f"company_name must not be null for any row, found {null_count} nulls"
print(f"Problem 5 passed ✓  ({cnt} rows)")